# Lesson 07: Convolutional Neural Networks (CNNs) & Classic Architectures

In this self-contained Colab notebook, you will:
1. Build a multi-layer **PyTorch CNN** (`nn.Conv2d`, `nn.MaxPool2d`, `nn.ReLU`) from scratch.
2. Train it on **CIFAR-10** (10 real-world object classes) on a Free Tier T4 GPU.
3. Complete the **Lab**: Classify multi-class animal images using a pyramid CNN.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_full = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

# Subset for fast free-tier execution; remove Subset() for full 50k training
train_loader = DataLoader(Subset(train_full, range(10000)), batch_size=64, shuffle=True)
test_loader = DataLoader(Subset(test_full, range(2000)), batch_size=128)

In [ ]:
class PyramidCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.flatten(1))

model = PyramidCNN().to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    print(f"Epoch {epoch+1} | CIFAR-10 Val Acc: {100 * correct / 2000:.2f}%")